# MSD RecSys — pipeline notebook

End-to-end driver for the Million Song Dataset two-stage recommender.
The library lives in `msd_recsys/`; this notebook just stitches the stages together
and prints diagnostics between them.

**Stages**
1. Setup (install, mount Drive, set paths)
2. Load + filter + split data
3. Stage 1 — retrieval (ALS + semantic IDs -> hybrid pool)
4. Stage 2 — features + ranker
5. Evaluation (overall + bucketed)
6. Submission


## 1. Setup


In [ ]:
# --- Colab-only: clone fresh + install. Safe to re-run. ---
import os
os.chdir('/content')                    # so we're not inside the dir we're about to delete
!rm -rf ecs172-recommender-systems
!git clone https://github.com/andrewscoding2018/ecs172-recommender-systems.git
%cd ecs172-recommender-systems
!pip install -q -e ./final_project

# --- Mount Drive for checkpoints (no-op if already mounted) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("not in Colab — running locally")

In [ ]:
# !pip uninstall -y implicit
# !pip install --no-cache-dir --no-binary=implicit implicit

In [ ]:
# from implicit.gpu import HAS_CUDA
# print(f"HAS_CUDA: {HAS_CUDA}")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from msd_recsys import data, retrieval, features, ranker, eval as ev, diagnostics as diag
from msd_recsys.checkpoint import checkpoint, set_checkpoint_dir, list_checkpoints

# Adjust these for your environment.
if IN_COLAB:
    DATA_DIR = Path('/content/drive/MyDrive/ECS 172')
    CKPT_DIR = Path('/content/drive/MyDrive/ECS 172/msd_checkpoints')
else:
    DATA_DIR = Path('./data')
    CKPT_DIR = Path('./checkpoints')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
set_checkpoint_dir(CKPT_DIR)
print(f"DATA_DIR = {DATA_DIR}")
print(f"CKPT_DIR = {CKPT_DIR}")
print(f"existing checkpoints: {list_checkpoints()}")


## 2. Load + filter + split


In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/ECS 172')

for p in sorted(DATA_DIR.iterdir()):
    print(f"  {p.name:50s}  {p.stat().st_size / 1e6:>10.1f} MB")

In [ ]:
# --- Big files: training interactions + metadata DB ---
interactions = checkpoint(
    "interactions_raw_v1",
    lambda: data.load_triplets(DATA_DIR / "train_triplets.txt"),
)
metadata = checkpoint(
    "metadata_v1",
    lambda: data.load_track_metadata(DATA_DIR / "track_metadata.db"),
)

# --- Canonical Kaggle files (small, no need to checkpoint) ---
# kaggle_songs.txt:                     canonical catalog (~386K song IDs)
# kaggle_users.txt:                     canonical user list (~110K user IDs we must predict for)
# kaggle_visible_evaluation_triplets:   visible half of the eval users' history (~1.45M rows)
canonical_songs = data.load_song_id_list(DATA_DIR / "kaggle_songs.txt")
canonical_users = data.load_user_list(DATA_DIR / "kaggle_users.txt")
visible_eval    = data.load_triplets(DATA_DIR / "kaggle_visible_evaluation_triplets.txt")

# taste_profile_song_to_tracks.txt is not strictly required — triplets and
# metadata.db both key on song_id, so we can join directly. Load it only if you
# have it (e.g., for cross-checking against per-track MSD audio files later).
song_to_track = None
if (DATA_DIR / "taste_profile_song_to_tracks.txt").exists():
    song_to_track = data.load_song_to_track(DATA_DIR / "taste_profile_song_to_tracks.txt")

print(f"interactions (train):    {interactions.shape}")
print(f"metadata:                {metadata.shape}")
print(f"canonical songs:         {len(canonical_songs):,}")
print(f"canonical users:         {len(canonical_users):,}")
print(f"visible eval triplets:   {visible_eval.shape}")
if song_to_track is not None:
    print(f"song-to-track mapping:   {song_to_track.shape}")
diag.diag_data(interactions, items_catalog=metadata)


## 2.5 EDA — preliminary results for the report

These cells produce figures and tables you can paste straight into `report.pdf`.
All figures save to `FIGURE_DIR` (Drive on Colab, `./figures/` locally) and also
display inline.


In [ ]:
# === EDA setup: matplotlib defaults + figure output path ===
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

FIGURE_DIR = (Path('/content/drive/MyDrive/msd_figures')
              if IN_COLAB else Path('./figures'))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"figures save to: {FIGURE_DIR}")


In [ ]:
# === EDA 1/5: dataset-at-a-glance summary table ===
n_interactions = len(interactions)
n_users        = interactions.user_id.nunique()
n_items_train  = interactions.song_id.nunique()
n_items_full   = len(metadata)
cold_share     = 1 - n_items_train / n_items_full

summary = pd.DataFrame([
    ("Interactions (train_triplets.txt)",     f"{n_interactions:,}"),
    ("Unique users in train",                  f"{n_users:,}"),
    ("Unique songs in train",                  f"{n_items_train:,}"),
    ("Songs in full catalog (metadata)",       f"{n_items_full:,}"),
    ("Cold-catalog share (no train listens)",  f"{cold_share:.1%}"),
    ("Mean listens / user",                    f"{n_interactions / n_users:.1f}"),
    ("Mean users / song (train items)",        f"{n_interactions / n_items_train:.1f}"),
    ("Canonical user list (kaggle_users)",     f"{len(canonical_users):,}"),
    ("Canonical song list (kaggle_songs)",     f"{len(canonical_songs):,}"),
], columns=["Statistic", "Value"])
print(summary.to_string(index=False))


In [ ]:
# === EDA 2/5: user history + song popularity distributions ===
hist_sizes = interactions.groupby("user_id").size()
pop_counts = interactions.groupby("song_id").size()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].hist(hist_sizes.clip(upper=200), bins=60, color="steelblue", edgecolor="white")
ax[0].axvline(hist_sizes.median(), color="red", ls="--",
              label=f"median = {hist_sizes.median():.0f}")
ax[0].set(xlabel="# train listens per user (clipped at 200)",
          ylabel="# users",
          title=f"User listening history sizes  (n={n_users:,})")
ax[0].legend()

ax[1].hist(np.log10(pop_counts), bins=60, color="darkorange", edgecolor="white")
ax[1].axvline(np.log10(pop_counts.median()), color="red", ls="--",
              label=f"median = {pop_counts.median():.0f}")
ax[1].set(xlabel="log10(# users who played the song)",
          ylabel="# songs",
          title=f"Song popularity  (n={n_items_train:,})")
ax[1].legend()

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig1_distributions.png")
plt.show()

print(f"\nUser history sizes:")
print(f"  median={hist_sizes.median():.0f}  p90={hist_sizes.quantile(0.9):.0f}  "
      f"p99={hist_sizes.quantile(0.99):.0f}  max={hist_sizes.max()}")
print(f"Song popularity:")
print(f"  median={pop_counts.median():.0f}  p90={pop_counts.quantile(0.9):.0f}  "
      f"p99={pop_counts.quantile(0.99):.0f}  max={pop_counts.max()}")


In [ ]:
# === EDA 3/5: long-tail / Pareto popularity curve (headline figure) ===
sorted_pop = pop_counts.sort_values(ascending=False).values
cum_share  = np.cumsum(sorted_pop) / sorted_pop.sum()
rank       = np.arange(1, len(sorted_pop) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Left: log-log rank vs listens (the classic long-tail shape)
ax[0].plot(rank, sorted_pop, color="darkorange", linewidth=1.5)
ax[0].set(xscale="log", yscale="log",
          xlabel="Song rank (most -> least popular)",
          ylabel="# users who played the song",
          title="Long-tail popularity curve  (log-log)")

# Right: Pareto cumulative share — "top X% = Y% of listens"
ax[1].plot(rank / len(rank) * 100, cum_share * 100,
           color="darkorange", linewidth=2)
top1_idx = int(0.01 * len(sorted_pop))
top1_pct = cum_share[top1_idx] * 100
ax[1].axvline(1, color="red", ls="--", alpha=0.6)
ax[1].axhline(top1_pct, color="red", ls="--", alpha=0.6)
ax[1].annotate(f"top 1% of songs\n= {top1_pct:.0f}% of listens",
               xy=(1, top1_pct), xytext=(8, top1_pct - 18),
               arrowprops=dict(arrowstyle="->", color="black"),
               fontsize=11)
ax[1].set(xlabel="% of catalog (sorted by popularity)",
          ylabel="cumulative % of listens",
          title="Pareto: how concentrated are listens?")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig2_longtail.png")
plt.show()

print("Top-X% of songs account for Y% of listens (motivates the diversity story):")
for pct in [1, 5, 10, 25, 50]:
    idx = max(int(pct/100 * len(sorted_pop)) - 1, 0)
    print(f"  top {pct:>3}%  ->  {cum_share[idx] * 100:>5.1f}% of listens")


In [ ]:
# === EDA 4/5: metadata completeness — justifies 'decade + unknown bucket' design ===
fields = ["title", "artist_id", "artist_name", "release",
          "year", "duration", "artist_familiarity", "artist_hotttnesss"]
fields = [f for f in fields if f in metadata.columns]
completeness = {f: metadata[f].notna().mean() * 100 for f in fields}

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = np.arange(len(completeness))
ax.barh(y_pos, list(completeness.values()),
        color="steelblue", edgecolor="white")
ax.set_yticks(y_pos)
ax.set_yticklabels(list(completeness.keys()))
ax.invert_yaxis()
ax.set_xlim(0, 102)
ax.set_xlabel("% of catalog items with non-missing value")
ax.set_title(f"Metadata completeness  (n={len(metadata):,} catalog items)")
for i, v in enumerate(completeness.values()):
    ax.text(v + 1, i, f"{v:.0f}%", va="center", fontsize=10)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig3_metadata_completeness.png")
plt.show()


In [ ]:
# === EDA 5/5: release year + decade distribution ===
year = metadata["year"].dropna()
year = year[(year >= 1950) & (year <= 2025)]  # drop obvious garbage

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Per-year histogram
ax[0].hist(year, bins=np.arange(1950, 2026, 1),
           color="steelblue", edgecolor="white")
ax[0].set(xlabel="Release year",
          ylabel="# songs",
          title=f"Release year distribution  "
                f"({len(year):,}/{len(metadata):,} = "
                f"{len(year)/len(metadata):.0%} have valid year)")

# Per-decade bars (this is the actual feature we use)
decade = (metadata["year"].dropna() // 10 * 10).astype(int)
decade = decade[(decade >= 1950) & (decade <= 2020)]
counts = decade.value_counts().sort_index()
ax[1].bar(counts.index.astype(str), counts.values,
          color="steelblue", edgecolor="white", width=0.7)
for i, (d, c) in enumerate(counts.items()):
    ax[1].text(i, c, f"{c:,}", ha="center", va="bottom", fontsize=9)
ax[1].set(xlabel="Decade bucket",
          ylabel="# songs",
          title="Songs per decade bucket (feature input)")
ax[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(FIGURE_DIR / "fig4_year_decade.png")
plt.show()

missing_year = metadata["year"].isna().sum() + (metadata["year"] == 0).sum()
print(f"missing or zero year: {missing_year:,} / {len(metadata):,} "
      f"({missing_year/len(metadata):.0%}) — these get an 'unknown' decade bucket")


In [ ]:
# --- Aggressive filter: drop tail items and low-activity users ---
MIN_SONG_LISTENS = 50
MIN_USER_LISTENS = 20

filtered = checkpoint(
    f"interactions_filtered_s{MIN_SONG_LISTENS}_u{MIN_USER_LISTENS}",
    lambda: data.filter_interactions(
        interactions,
        min_song_listens=MIN_SONG_LISTENS,
        min_user_listens=MIN_USER_LISTENS,
    ),
)
diag.diag_filter(interactions, filtered)


In [ ]:
# --- Hold-out split mirroring the MSD challenge format ---
KEEP_LAST_N = 5
train_inner, valid = checkpoint(
    f"split_n{KEEP_LAST_N}",
    lambda: data.holdout_split(filtered, n_per_user=KEEP_LAST_N),
)
print(f"train_inner: {len(train_inner):,} rows, {train_inner.user_id.nunique():,} users")
print(f"valid      : {len(valid):,} rows,    {valid.user_id.nunique():,} users")


In [ ]:
# --- Build sparse user-item matrix for ALS ---
ui_matrix, user_to_ix, item_to_ix = checkpoint(
    "ui_matrix_v1",
    lambda: data.build_user_item_matrix(train_inner, confidence_alpha=40.0),
)

print(f"user-item matrix: {ui_matrix.shape}, nnz={ui_matrix.nnz:,}")


## 3. Stage 1 — Retrieval


In [ ]:
# --- ALS via implicit (GPU-accelerated on A100 if available) ---
als = checkpoint(
    "als_f64_r0.01_i30",
    lambda: retrieval.ALSRetriever(factors=64, regularization=0.01, iterations=30).fit(
        ui_matrix, user_to_ix, item_to_ix,
    ),
)

# Optional sanity check — pass a few well-known song titles in if you have them.
# Build a title lookup from metadata for prettier diagnostic output.
title_by_song = dict(zip(metadata.song_id, metadata.title))
diag.diag_als(als, item_titles=title_by_song)


In [ ]:
# --- Semantic-ID retriever (metadata-only for now; revisit if adding audio) ---
# Filter metadata to the items that appear in our filtered training set so the
# semantic-ID space matches what we recommend over.
items_in_train = set(train_inner.song_id.unique())
metadata_in_train = metadata[metadata.song_id.isin(items_in_train)].copy()

sid = checkpoint(
    "sid_metadata_v1",
    lambda: retrieval.SemanticIDRetriever().fit(metadata_in_train),
)
diag.diag_semantic_codes(sid, item_titles=title_by_song, n_show=8)


In [ ]:
# === CELL A: setup (cheap, instant) ===
TOPK_ALS = 1500
TOPK_SID = 1500

valid_users = sorted(valid.user_id.unique())
valid_users = [u for u in valid_users if u in user_to_ix]
valid_user_indices = np.array([user_to_ix[u] for u in valid_users])
hist_by_user = data.histories_from_df(train_inner)
valid_histories = [hist_by_user.get(u, []) for u in valid_users]
valid_truth = valid.groupby("song_id" if False else "user_id")["song_id"].apply(set).to_dict()

print(f"valid users to retrieve for: {len(valid_users):,}")

In [ ]:
# === CELL A: setup + sampling (do sampling here, not later) ===
TOPK_ALS = 1500
TOPK_SID = 1500

valid_users = sorted(valid.user_id.unique())
valid_users = [u for u in valid_users if u in user_to_ix]

# ▼ Apply the cap here, so EVERY downstream cell sees the same N users
MAX_USERS = 80000
valid_users = valid_users[:MAX_USERS]
print(f"capped valid_users at {len(valid_users):,}")

valid_user_indices = np.array([user_to_ix[u] for u in valid_users])
hist_by_user = data.histories_from_df(train_inner)
valid_histories = [hist_by_user.get(u, []) for u in valid_users]
valid_truth = valid.groupby("user_id")["song_id"].apply(set).to_dict()

In [ ]:
# === CELL B: ALS retrieval (potentially slow, then cached) ===
als_ids, als_sc = checkpoint(
    f"als_topk_v_{TOPK_ALS}",
    lambda: als.recommend_batch(
        ui_matrix, top_k=TOPK_ALS, user_indices=valid_user_indices,
        verbose=True,
    ),
)

In [ ]:
import psutil
print(f"RAM total:     {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB")
print(f"len(valid_users)  = {len(valid_users):,}")
print(f"TOPK_ALS, TOPK_SID = {TOPK_ALS}, {TOPK_SID}")
print(f"len(sid.all_items) = {len(sid.all_items):,}  <- SID catalog size")
print(f"als_ids in memory? {('als_ids' in dir())}  shape={getattr(als_ids, 'shape', '—')}")

In [ ]:
# === CELL C: SID retrieval, memory-paranoid version ===
import gc
import time
from tqdm.auto import tqdm

# Hard cap on users to retrieve for, to ensure this finishes.
MAX_USERS = 5000
hist_subset = valid_histories[:MAX_USERS]
print(f"running SID for {len(hist_subset):,} users (capped from {len(valid_histories):,})")

def sid_retrieve_safe():
    n = len(hist_subset)
    # int32 output regardless of library version — way smaller than object dtype.
    ids_out = np.empty((n, TOPK_SID), dtype=np.int32)
    sc_out  = np.empty((n, TOPK_SID), dtype=np.float32)
    CHUNK = 500  # smaller chunk = more iterations, less peak memory

    for s in tqdm(range(0, n, CHUNK), desc="SID batches", unit="batch"):
        e = min(s + CHUNK, n)
        chunk_hist = hist_subset[s:e]
        ids, sc = sid.recommend_for_histories(
            chunk_hist, top_k=TOPK_SID,
            already_owned=[set(h) for h in chunk_hist],
        )
        # If lib still returns object dtype, coerce to int via lookup.
        if ids.dtype == object:
            # Map song_id strings -> indices using the SID retriever's mapping
            lookup = sid._item_to_ix
            ids_int = np.empty(ids.shape, dtype=np.int32)
            for i in range(ids.shape[0]):
                for j in range(ids.shape[1]):
                    v = ids[i, j]
                    ids_int[i, j] = lookup.get(v, -1) if v is not None else -1
            ids = ids_int
        ids_out[s:e] = ids
        sc_out[s:e]  = sc

        # Free intermediates and report
        del ids, sc, chunk_hist
        gc.collect()
        avail = psutil.virtual_memory().available / 1e9
        if avail < 3.0:
            print(f"  ⚠️  only {avail:.1f} GB available — consider lowering TOPK_SID or MAX_USERS")

    return ids_out, sc_out

t = time.time()
sid_ids, sid_sc = checkpoint(f"sid_topk_v_{TOPK_SID}_n{MAX_USERS}", sid_retrieve_safe)
print(f"\nSID done: {time.time() - t:.1f}s  shape={sid_ids.shape}  "
      f"({sid_ids.nbytes / 1e6:.1f} MB)")

In [ ]:
# === CELL D: hybrid candidate pool (long-format DataFrame) ===
# Build the canonical item index space from metadata (full catalog).
canonical_item_to_ix = {it: i for i, it in enumerate(metadata_in_train.song_id.values)}

valid_cands = retrieval.build_candidate_pool(
    als_ids, als_sc, als.ix_to_item,
    sid_ids, sid_sc, sid.all_items,
    canonical_item_to_ix,
    verbose=True,
)
# valid_cands is now a DataFrame, not a list of dicts!
print(valid_cands.head())
print(f"avg pool size: {len(valid_cands) / len(valid_users):.1f}/user")

# diag.diag_pool currently expects list-of-dicts — temporarily skip or
# rewrite to consume the DataFrame. Quick replacement:
n_users = len(valid_users)
n_als_only = ((valid_cands.als_score > 0) & (valid_cands.sid_score == 0)).sum()
n_sid_only = ((valid_cands.als_score == 0) & (valid_cands.sid_score > 0)).sum()
n_both     = ((valid_cands.als_score > 0) & (valid_cands.sid_score > 0)).sum()
total = len(valid_cands)
print(f"ALS only      : {n_als_only/n_users:8.1f}/user ({n_als_only/total:.1%})")
print(f"semantic only : {n_sid_only/n_users:8.1f}/user ({n_sid_only/total:.1%})")
print(f"both routes   : {n_both/n_users:8.1f}/user ({n_both/total:.1%})")

## 4. Stage 2 — Features + ranker


In [ ]:
# --- Build item feature table (one-shot, used by every (user, candidate) row) ---
item_pop = train_inner.groupby("song_id").size()
item_table = features.ItemFeatureTable.build(
    metadata=metadata_in_train,
    item_popularity=item_pop,
    semantic_codes=sid.item_codes,
)
print(f"item feature table: {len(item_table.item_ids):,} items")


In [ ]:
# --- 80/20 split of validation users for ranker fit vs eval ---
rng = np.random.default_rng(42)
perm = rng.permutation(len(valid_users))
split = int(0.8 * len(valid_users))
fit_ix, eval_ix = perm[:split], perm[split:]

def slice_(arr, ixs): 
    return [arr[i] for i in ixs]
fit_users  = slice_(valid_users,     fit_ix)
fit_hist   = slice_(valid_histories, fit_ix)
fit_cands  = slice_(valid_cands,     fit_ix)
eval_users = slice_(valid_users,     eval_ix)
eval_hist  = slice_(valid_histories, eval_ix)
eval_cands = slice_(valid_cands,     eval_ix)

X_fit,  y_fit,  k_fit,  g_fit  = features.build_feature_rows(
    fit_users, fit_hist, fit_cands, item_table, truth_by_user=valid_truth,
)
X_eval, y_eval, k_eval, g_eval = features.build_feature_rows(
    eval_users, eval_hist, eval_cands, item_table, truth_by_user=valid_truth,
)

# Drop zero-sized groups (users whose candidate dicts came back empty)
g_fit_nz  = [g for g in g_fit  if g > 0]
g_eval_nz = [g for g in g_eval if g > 0]
print(f"ranker fit : X={X_fit.shape}, positives={int(y_fit.sum()):,} ({y_fit.mean():.2%})")
print(f"ranker eval: X={X_eval.shape}, positives={int(y_eval.sum()):,} ({y_eval.mean():.2%})")
diag.diag_features(X_fit, y_fit, features.FEATURE_NAMES)


In [ ]:
# --- Train LightGBM lambdarank ---
model = ranker.train_lambdarank(
    X_fit, y_fit, group_sizes=g_fit_nz,
    eval_set=(X_eval, y_eval), eval_group=g_eval_nz,
    n_estimators=300, max_position=500, early_stopping_rounds=20,
)
diag.diag_permutation_importance(model, X_eval, y_eval, features.FEATURE_NAMES, rng=rng)


## 5. Evaluation


In [ ]:
# --- Rank candidates -> top-500 per user, compute metrics ---
eval_owned = {u: set(h) for u, h in zip(eval_users, eval_hist)}
ranked = ranker.rank_candidates(model, X_eval, k_eval, top_k=500, exclude_owned=eval_owned)

map500    = ev.map_at_k(ranked,    {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)
recall500 = ev.mean_recall_at_k(ranked, {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)
ndcg500   = ev.mean_ndcg_at_k(ranked,   {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)

# Retrieval ceiling on the eval slice
ret_recall = ev.retrieval_recall(
    {u: set(c.keys()) for u, c in zip(eval_users, eval_cands)},
    {u: valid_truth[u] for u in eval_users if u in valid_truth},
)

# Sum-of-scores baseline
baseline_scores = X_eval[:, features.FEATURE_NAMES.index("als_score")] \
                + X_eval[:, features.FEATURE_NAMES.index("semantic_score")]
baseline_ranked: dict = {}
for (u, it), s in zip(k_eval, baseline_scores):
    baseline_ranked.setdefault(u, []).append((it, float(s)))
for u, lst in baseline_ranked.items():
    lst.sort(key=lambda x: -x[1])
    owned = eval_owned.get(u, set())
    baseline_ranked[u] = [it for it, _ in lst if it not in owned][:500]
baseline_map500 = ev.map_at_k(baseline_ranked, {u: valid_truth[u] for u in eval_users if u in valid_truth}, k=500)

print(f"MAP@500     : {map500:.4f}")
print(f"Recall@500  : {recall500:.4f}")
print(f"NDCG@500    : {ndcg500:.4f}")
print(f"baseline MAP@500 (als+sid sum): {baseline_map500:.4f}")
diag.diag_results(map_score=map500, retrieval_recall=ret_recall, baseline_map=baseline_map500)


In [ ]:
# --- Bucketed evaluation: by user history size ---
user_hist_size = {u: len(h) for u, h in zip(eval_users, eval_hist)}
def history_bucket(n):
    if n < 30:   return "0_light"     # leading underscores for sort order
    if n < 100:  return "1_medium"
    return "2_heavy"
user_buckets = {u: history_bucket(n) for u, n in user_hist_size.items()}

bucketed = ev.metrics_by_bucket(ranked, {u: valid_truth[u] for u in eval_users}, user_buckets, k=500)
print("by user-history bucket:")
print(bucketed.to_string(index=False))


In [ ]:
# --- Catalog coverage + tail-focused MAP ---
catalog_size = len(item_table.item_ids)
coverage = ev.catalog_coverage(ranked, catalog_size, k=500)
print(f"catalog coverage @500: {coverage:.2%}  ({int(coverage * catalog_size):,} of {catalog_size:,} items recommended to >=1 user)")

# Popularity tiers (head/torso/tail by cumulative listen share)
item_tier = ev.popularity_tiers(item_pop)
tail_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="tail")
torso_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="torso")
head_map = ev.tail_focused_map(ranked, {u: valid_truth[u] for u in eval_users}, item_tier, k=500, tier="head")
print(f"\nMAP@500 by ground-truth popularity tier:")
print(f"  head : {head_map:.4f}")
print(f"  torso: {torso_map:.4f}")
print(f"  tail : {tail_map:.4f}")
print("  -> the tail number is the 'discovery' signal — does the system help users find long-tail songs they actually played?")


In [ ]:
truth_items = set()
for u in valid_users_s:
    truth_items |= valid_truth.get(u, set())

train_items = set(train_inner.song_id.unique())
sid_items   = set(sid.all_items)

print(f"unique held-out items     : {len(truth_items):,}")
print(f"  also scorable by ALS    : {len(truth_items & train_items):,}  ({len(truth_items & train_items)/len(truth_items):.1%})")
print(f"  also scorable by SID    : {len(truth_items & sid_items):,}  ({len(truth_items & sid_items)/len(truth_items):.1%})")
print(f"  unreachable by EITHER   : {len(truth_items - train_items - sid_items):,}  ({len(truth_items - train_items - sid_items)/len(truth_items):.1%})")

In [ ]:
hist_sizes = pd.Series([len(h) for h in valid_histories_s])
print(f"valid history sizes (after last-5 holdout):")
print(f"  min/p25/median/p75/max: {hist_sizes.min()} / {hist_sizes.quantile(.25):.0f} / {hist_sizes.median():.0f} / {hist_sizes.quantile(.75):.0f} / {hist_sizes.max()}")
print(f"  users with ≤5 listens : {(hist_sizes <= 5).sum():,} ({(hist_sizes <= 5).mean():.0%})")

In [ ]:
pop = train_inner.groupby("song_id").size().sort_values(ascending=False)
pop_top = pop.head(2000).index.tolist()  # match your ~1922 pool size

# How many held-out hits would top-2000-by-popularity catch?
pop_set = set(pop_top)
hits = total = 0
for u, t in valid_truth.items():
    if u not in set(valid_users_s):
        continue
    if not t:
        continue
    hits += len(t & pop_set)
    total += len(t)
print(f"popularity-baseline recall@2000: {hits/total:.4f}")

## 6. Submission


In [ ]:
# --- Retrain Stage 1 on full filtered data + rank for test users ---
# Skeleton — adapt once you've confirmed which hidden test file to evaluate against.

# 1. Rebuild ui_matrix on `filtered` (not train_inner), refit ALS, refit SID.
# 2. Load test user_ids from kaggle_users.txt or the canonical hidden file.
# 3. For each test user, retrieve TOPK_ALS + TOPK_SID, build features, rank to top-500.
# 4. Format CSV per the MSD challenge format and validate.

# Placeholder — uncomment and adapt:
# test_users = data.load_user_list(DATA_DIR / "kaggle_users.txt")
# ... retrieve + rank ...
# submission.to_csv('submission.csv', index=False)
print("submission cell is a skeleton — fill in once test format is confirmed.")
